# Lab 4.1 &mdash; OpenCode to Jira, over MCP

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Tool Calling &amp; MCP**

### What you'll do
- Talk to a real MCP server by hand &mdash; <code>initialize</code>, then <code>tools/list</code>
- Grant an agent access to Jira by writing four lines of JSON
- Watch <code>opencode</code> triage a payment exception and raise the ticket for it
- See what the grant actually cost you &mdash; in tools, in context and in audit trail

> **How this lab works &mdash; it is different from the others.** There is nothing to fill in
> and nothing to score. You run the cells in order and watch a real agent reach a real Jira
> over MCP. The participant notebook and the solution notebook are the same file, on purpose:
> the point is to *see the protocol work* before Module 4 asks you to build one. Read the
> output of every cell &mdash; that is the lab.

> **Nothing to fill in.** This is the one lab in Module 4 you only *run*. Everything
> after it asks you to build; this one asks you to look.

## The use case

You are on a payments operations desk. A payment lands in an exception queue and somebody has to
decide what happens to it: read the record, check it against policy, and &mdash; if it needs a
human &mdash; raise a ticket with enough detail that the next person does not start from nothing.

The reading and the deciding are what an agent is good at. The ticket is the part that touches a
system you do not own: **Jira**, run by another team, with its own credentials and its own audit
trail.

Without MCP you would write a Jira client, an auth flow, a schema for every call, and then do it
again for the next agent. With MCP the Jira team publishes one server, and every agent &mdash;
yours, Claude Code, Cursor, the next one &mdash; speaks to it the same way.

That is what you are about to do, end to end, in about fifteen minutes of running time.

## Before you start

Two environment variables are already set in your sandbox:

| variable | what it is |
|---|---|
| `JIRA_MCP_URL` | the Jira MCP server the class shares &mdash; one server, everyone's agent |
| `JIRA_MCP_AUTH` | the credential your agent presents to it |

Run the next cell. If it reports something missing, ask the trainer &mdash; do not go looking for
a token, and do not paste one into a notebook.

In [ ]:
# ------------------------------------------------------------ Preflight: run me first
import os, re, json, socket, subprocess, textwrap, urllib.request, urllib.error

MCP_URL  = os.environ.get("JIRA_MCP_URL", "")
MCP_AUTH = os.environ.get("JIRA_MCP_AUTH", "")
PROJECT  = os.environ.get("JIRA_MCP_PROJECT", "MCPLAB")
# Who you are, for tagging tickets on a board the whole class shares. The pod hostname
# is the only reliable source here: JUPYTERHUB_USER and USER are both unset in the
# sandbox, and everyone is the OS user "jovyan".
def _whoami() -> str:
    host = socket.gethostname()                  # e.g. "agenticaiu31-0"
    m = re.search(r"(u\d+)", host)
    return m.group(1) if m else (os.environ.get("JUPYTERHUB_USER") or host or "u0")

WHO      = _whoami()
LABDIR   = os.path.expanduser("~/work/mcplab")          # home, not /tmp: /tmp is wiped on restart

def ready() -> bool:
    """True when the sandbox has everything this lab needs."""
    return bool(MCP_URL and MCP_AUTH)

if ready():
    print("MCP server :", MCP_URL)
    print("credential : present (%d chars, not shown)" % len(MCP_AUTH))
    print("project    :", PROJECT)
    print("you are    :", WHO)
    os.makedirs(LABDIR, exist_ok=True)
    print("lab folder :", LABDIR)
else:
    print("Not configured yet. This lab needs two variables that the sandbox should already have:")
    for name in ("JIRA_MCP_URL", "JIRA_MCP_AUTH"):
        print(f"  {name:14} {'set' if os.environ.get(name) else 'MISSING'}")
    print("\nEvery cell below will skip cleanly until they are set. Ask the trainer.")

## Step 1 &mdash; Meet the server without an agent

Before any model is involved, talk to the server yourself. This is the lifecycle from the deck,
on the wire:

1. **`initialize`** &mdash; agree a protocol version, exchange capabilities and identity
2. **`tools/list`** &mdash; discovery: the client learns what exists, at run time

Nothing here is Jira-specific. Any MCP server answers these two calls the same way, which is the
entire point of a protocol.

In [ ]:
def rpc(method: str, params: dict | None = None, sid: str | None = None):
    """One JSON-RPC call to the MCP server over streamable HTTP.

    Returns (result, session_id). The Authorization header is the whole of our
    credential: the server decides what we may do purely from that.
    """
    body = json.dumps({"jsonrpc": "2.0", "id": 1, "method": method,
                       "params": params or {}}).encode()
    req = urllib.request.Request(MCP_URL, data=body, method="POST")
    req.add_header("Content-Type", "application/json")
    req.add_header("Accept", "application/json, text/event-stream")
    req.add_header("Authorization", "Basic " + MCP_AUTH)
    if sid:
        req.add_header("Mcp-Session-Id", sid)
    with urllib.request.urlopen(req, timeout=60) as r:
        raw, sess = r.read().decode(), r.headers.get("mcp-session-id")
    # streamable HTTP may frame the reply as an SSE event; take the data line either way
    for line in raw.splitlines():
        if line.startswith("data:"):
            raw = line[5:].strip()
            break
    return json.loads(raw).get("result", {}), sess


if ready():
    init, session = rpc("initialize", {
        "protocolVersion": "2025-06-18",
        "capabilities": {},
        "clientInfo": {"name": f"lab-4-1-{WHO}", "version": "1.0"},
    })
    print("protocolVersion :", init.get("protocolVersion"))
    print("serverInfo      :", init.get("serverInfo"))
    print("capabilities    :", ", ".join(init.get("capabilities", {})))
    print("session         :", session)
else:
    print("skipped - see the preflight cell")

Three things worth pausing on in that reply.

- **`protocolVersion`** is agreed, not assumed. The client proposed one; the server answered with
  the version it will actually speak.
- **`serverInfo`** is the server naming itself &mdash; and the MCP spec is explicit that this is
  *self-reported and unverified*. It is for display and logging. Never make a security decision on
  it.
- **`capabilities`** is the server saying what it supports before you use any of it.

Now discovery. Your client did not know a single tool name a moment ago.

In [ ]:
if ready():
    tools, _ = rpc("tools/list", {}, sid=session)
    names = [t["name"] for t in tools.get("tools", [])]
    print(f"the server published {len(names)} tools\n")
    for n in sorted(names)[:12]:
        print("  -", n)
    print("  ... and", max(0, len(names) - 12), "more")

    example = next((t for t in tools["tools"] if t["name"] == "jira_create_issue"), tools["tools"][0])
    print("\nwhat the model actually reads for one of them:\n")
    print("  name        :", example["name"])
    print("  description :", textwrap.shorten(example.get("description", ""), 150))
    print("  inputSchema :", ", ".join(list(example.get("inputSchema", {}).get("properties", {}))[:8]), "...")
else:
    print("skipped - see the preflight cell")

**Name, description, inputSchema.** That is the whole of what reaches the model &mdash; the same
three fields Module 4 keeps coming back to, except this time you did not write them. The Jira team
did, and your agent's accuracy now depends on their prose.

Notice the tool *count*: a handful, not everything Jira can do. That is deliberate, and the last
section explains what it is protecting you from.

## Step 2 &mdash; The config file is the grant

Now hand the server to an agent. `opencode` reads an `opencode.json` from the folder it runs in;
this is the whole integration.

Two details that matter more than they look:

- **`"type": "remote"`** &mdash; this server is not a subprocess we launched. It runs elsewhere,
  serves the whole class, and holds the Jira credentials so that we do not have to.
- **`{env:JIRA_MCP_AUTH}`** &mdash; the token is read from the environment at run time. It is
  never written into the file, which is why this file can live in a public repository.

In [ ]:
CONFIG = {
    "$schema": "https://opencode.ai/config.json",
    # the lab gateway, registered under its own name so it is unaffected by any
    # provider the sandbox has disabled by default
    "provider": {
        "litellm": {
            "npm": "@ai-sdk/openai-compatible",
            "name": "LiteLLM Gateway",
            "options": {"baseURL": "{env:LAB_LLM_BASE_URL}", "apiKey": "{env:LITELLM_API_KEY}"},
            "models": {"qwen36-35b-a3b-lab": {"name": "Qwen3.6 35B A3B (lab)"}},
        }
    },
    "mcp": {
        "jira": {
            "type": "remote",
            "url": "{env:JIRA_MCP_URL}",
            "enabled": True,
            "headers": {"Authorization": "Basic {env:JIRA_MCP_AUTH}"},
        }
    },
}

if ready():
    path = os.path.join(LABDIR, "opencode.json")
    with open(path, "w") as fh:
        json.dump(CONFIG, fh, indent=2)
    print("wrote", path, "\n")
    print(open(path).read())
else:
    print("skipped - see the preflight cell")

## Step 3 &mdash; Confirm the connection

`opencode mcp list` asks every configured server to prove it is there.

If this says **needs authentication** rather than **connected**, the `Authorization` header is not
arriving &mdash; the server answers `401`, and `opencode` offers you an OAuth flow this server does
not implement. That is a missing environment variable, not a broken server.

In [ ]:
def oc(*args, timeout=300):
    """Run a SHORT opencode command from the notebook and return its output.

    Used for `mcp list` only. Full agent turns (`opencode run`) go in a terminal --
    they stream, they take minutes, and watching the tool calls scroll past is most of
    the point.
    """
    p = subprocess.run(["opencode", *args], cwd=LABDIR, capture_output=True,
                       text=True, timeout=timeout)
    out = (p.stdout or "") + (p.stderr or "")
    return re.sub(r"\x1b\[[0-9;?]*[a-zA-Z]", "", out)       # strip the spinner escapes


if ready():
    print(oc("mcp", "list", timeout=120))
else:
    print("skipped - see the preflight cell")

## Step 4 &mdash; Let it read

The first real turn &mdash; and this one you run **in the terminal**, not in the notebook.

`opencode` is a terminal agent: it streams its thinking, shows each tool call as it happens, and
that is the part worth watching. Run the next cell to print the command, then open a terminal in
JupyterLab (**File &rarr; New &rarr; Terminal**) and paste it.

Expect roughly a minute. The agent has to discover the tools, choose one, and shape the arguments
from your sentence.

In [ ]:
READ_TASK = (
    f"Use the jira MCP tools. Search project {PROJECT} and tell me how many issues it has, "
    "then list up to five of their keys and summaries. Do not create or modify anything."
)

def terminal_command(task: str) -> str:
    """The exact line to paste into a JupyterLab terminal."""
    return (f"cd {LABDIR} && \\\n"
            f'  opencode run --model litellm/qwen36-35b-a3b-lab \\\n    "{task}"')

if ready():
    print("Open File > New > Terminal, then paste:\n")
    print(terminal_command(READ_TASK))
else:
    print("skipped - see the preflight cell")

Watch the terminal as it answers. You should see a line beginning `⚙` &mdash; that is the agent
calling an MCP tool, with the arguments it chose. Nothing in your sentence named a tool.

## Step 5 &mdash; Let it write

Reading is reassuring. Writing is the point: this is the moment the agent stops being a chat
window and starts changing a system of record.

The summary is tagged with your sandbox name so you can find your own ticket on a shared board.

In [ ]:
WRITE_TASK = (
    f"Use the jira MCP tools. Create ONE issue in project {PROJECT}, issue type Task, "
    f"with the summary exactly: [{WHO}] Payment PMT-1003 held for manual review. "
    "Give it a one-line description explaining that the payment breached the review threshold "
    "and needs an operator decision. Then reply with only the new issue key."
)

if ready():
    print("Same terminal, next command:\n")
    print(terminal_command(WRITE_TASK))
else:
    print("skipped - see the preflight cell")

## Step 6 &mdash; Check it independently

Never take the agent's word for a write. Ask Jira, through the same MCP server but without a model
in the loop &mdash; `tools/call` is the third method from the deck, and it is just another JSON-RPC
call.

In [ ]:
if ready():
    found, _ = rpc("tools/call", {
        "name": "jira_search",
        "arguments": {"jql": f'project = {PROJECT} ORDER BY created DESC', "limit": 10},
    }, sid=session)
    text = "".join(c.get("text", "") for c in found.get("content", []))
    mine = [ln for ln in text.splitlines() if WHO in ln]
    print("lines mentioning you:\n")
    print("\n".join(mine) if mine else "(none yet - re-run Step 5)")
    print("\n--- raw, first 600 chars ---\n")
    print(text[:600])
else:
    print("skipped - see the preflight cell")

## What MCP actually bought you

Look back at what you wrote: **a JSON object with four keys**. No Jira SDK, no auth code, no
request signing, no schema for `jira_create_issue`, no retry logic. The integration was a
configuration change.

| without MCP | what you just did |
|---|---|
| write a Jira client for this agent | write four lines of JSON |
| repeat it for the next agent | the next agent reuses the same server |
| hold Jira credentials in the agent | the server holds them; you send one header |
| pin to a Jira API version in your code | the server publishes its tools at run time |

And the same server is already serving everyone else in this room, right now, from their own
sandbox.

## What it also cost you &mdash; three things to carry into Module 4

**1. Tool count is context, and it has a breaking point.** The server you just used publishes
five tools. The same software, unscoped, publishes **sixty-three** &mdash; everything Jira can do,
including sprints, worklogs and attachments. Every one of those schemas is sent to the model on
*every* turn, before your question is even read.

That is not a theoretical cost. This lab was built twice. With sixty-three tools the agent failed
outright on this model &mdash; connection fine, discovery fine, and then the turn died the moment
the tool schemas were in front of it. With five it does the job first time, which is the run you
just watched. The server was scoped with one flag:

```
--enabled-tools jira_search,jira_get_issue,jira_create_issue,jira_add_comment,jira_get_project_issues
```

Scoping a server to the tools an agent actually needs is not tidying. It is what makes the agent
work at all.

**2. The descriptions are not yours.** Your agent picked `jira_create_issue` over the
alternatives because of a sentence someone on the Jira team wrote, and shaped its arguments from a
schema they published. When selection goes wrong, the fix may live in a repository you cannot
commit to.

**3. One credential, one identity.** Every agent in this room authenticated as the *same* service
account, so Jira's audit trail will show one name against thirty people's work. That is a
deliberate simplification for a classroom. In production the identity on the credential is the
identity in the audit log &mdash; which is exactly why the config file deserves the same review as
an IAM policy.

## Your turn

Nothing here is graded. Try a couple and watch which ones the agent gets right:

- Ask it to **add a comment** to the issue it just created.
- Ask it to **find every issue mentioning PMT-1003** and summarise them in one line.
- Ask for something the server **cannot** do &mdash; delete the issue, say. The tool was scoped
  out, so it is not that permission was denied: from the agent's side the capability simply does
  not exist. Read how that failure reads compared with a policy refusal.
- Open `opencode.json` and set `"enabled": false`. Re-run Step 4 and watch the same sentence
  produce a completely different answer. That single flag is the grant.

## Cleanup

Nothing to clean up in your sandbox &mdash; the server is not yours and the config is a file in
your home directory. Your ticket stays on the board; that is the evidence it worked.

Next: **Lab 4.2**, where the tools stop being someone else's and become yours.